# Tagging Automatique

Ici l'objectif est de pouvoir attribuer un tag aux paroles de chansons.

In [1]:
from function.utils import os, tqdm, json, glob

PATH_JSON = "../data/json_file/"

dictionnary3 = PATH_JSON+"dictionnaire_song.json"
dictionnary_kaggle = PATH_JSON+"dictionnaire_data_kaggle.json"

In [2]:
with open(dictionnary3, "r") as f:
    dict3 = json.load(f)

with open(dictionnary_kaggle, "r") as f:
    dict_kaggle = json.load(f)

In [3]:
genre = ["pop", "rap", "classique"]
cpt = 0


for artiste, songs in dict3.items():
    artistes_a_supprimer = []
    for titre, values in songs.items():
        if values.get("completion") is None:
            artistes_a_supprimer.append(titre)

        if cpt == 3:
            cpt = 0
        new_prompt = values["prompt"]+ f" {genre[cpt]}"

        dict3[artiste][titre]['prompt'] = new_prompt

    cpt += 1

    for titre in artistes_a_supprimer:
        del dict3[artiste][titre]
        

In [4]:
new_dict_kaggle = {}

for dico in dict_kaggle:
    artist = dico["artist"]
    title = dico["title"]
    genre = dico["topic_clean"]
    lyrics = dico["lyrics"]

    new_dict_kaggle.setdefault(artist, {})
    new_dict_kaggle[artist].setdefault(title, {})

    new_dict_kaggle[artist][title]["prompt"] = (
        f"Artiste: {artist}\n"
        f"Titre: {title}\n"
        f"Genre: {genre}"
    )
    
    new_dict_kaggle[artist][title]["completion"] = lyrics



In [5]:
dict_fusionner = dict3.copy()

for artist, songs in new_dict_kaggle.items():

    if artist not in dict_fusionner:
        dict_fusionner[artist] = songs
    else:
        dict_fusionner[artist].update(songs)



In [6]:
for artiste, songs in dict_fusionner.items():
    artiste_to_delete = []
    if len(songs) == 0:
        artiste_to_delete.append(dict_fusionner[artiste])
for artiste in artiste_to_delete:
    del dict_fusionner[artiste]

In [7]:
# 💾 Sauvegarde
with open(PATH_JSON + "final_dict_song.json", "w", encoding="UTF-8") as json_file:
    json.dump(dict_fusionner, json_file, indent=4, ensure_ascii=False)